# LELA60331 Week 2 Seminar
This week we are going to be thinking about and working with n-gram language models, practising some Python skills along the way.

We start by building a **character-based language model** from a sample text. Then you will play the **Shannon game** (to be explained), and finally we will use both to estimate the **entropy of English** and compare the results.

## A character-based language model

Our sample text is the first chapter of *Crime and Punishment*. We will count how often each character, and each pair of characters, occurs in it, and use these counts to assign probabilities to characters and to whole sentences.

In [ ]:
# download the book from Project Gutenberg (the ! runs a command outside Python)
!wget https://www.gutenberg.org/files/2554/2554-0.txt
# read the whole file into one string
with open('2554-0.txt') as fp:
    c_and_p = fp.read()
# find where chapter one starts and ends: from just after the heading 'CHAPTER I'
# up to the heading 'CHAPTER II' (the Week 3 seminar looks at this more closely)
start = c_and_p.find('CHAPTER I\n') + len('CHAPTER I\n')
end = c_and_p.find('CHAPTER II\n')
# select chapter one and convert it to lower case
raw = c_and_p[start:end].lower()
# keep only the letters a-z and the space, using a for loop
allowed = 'abcdefghijklmnopqrstuvwxyz '
kept = []
for ch in raw:
    if ch in allowed:
        kept.append(ch)
chapter_one = ''.join(kept)       # glue the kept characters back into one string

In [ ]:
chapter_one

**Activity 1 (counting with a string method)**

The first line counts how many times `'the'` occurs. Fill in the second line to count `' the '`, with a space either side. Why are the two numbers different?

In [ ]:
print(chapter_one.count('the'))
print(chapter_one.count(' the '))

We are going to be assigning probabilities to sentences using bigram models, so we need to extract unigram and bigram counts. We will store them in dictionaries.

In [ ]:
# Count how many times each character occurs, using the dictionary counting pattern from the Week 3 Python workbook
unigrams = {}
for ch in chapter_one:
    unigrams[ch] = unigrams.get(ch, 0) + 1

# the total number of characters
total_unigrams = len(chapter_one)

In [ ]:
unigrams

In [ ]:
total_unigrams

**Activity 2 (counting with if/else)**

The counting cell above counts characters with `unigrams.get(ch, 0) + 1`. Here is the same idea written with `if`/`else`: if we have seen the character before, add one to its count; otherwise this is the first time, so its count is 1.

Fill in the two blanks. The cell should print `True`.

In [ ]:
my_counts = {}
for ch in chapter_one:
    if ch in my_counts:
        my_counts[ch] += 1     # add one to the count we already have
    else:
        my_counts[ch] = 1     # first time we have seen this character

print(my_counts == unigrams)

**Activity 3 (probabilities of characters)**

The probability of a character is its count divided by the total number of characters. The first two lines work out P(e) as an example.

1. Work out P(t) and P(space) in the same way. (The space character is written `' '`.)
2. The probability of a vowel is P(a) + P(e) + P(i) + P(o) + P(u). Fill in the blank in the loop.
3. The probability of **not** getting a space is 1 − P(space).

In [ ]:
p_e = unigrams['e'] / total_unigrams
print('P(e) =', round(p_e, 3))

# 1.
p_t = unigrams['t']/total_unigrams
p_space = unigrams[' ']/total_unigrams
print('P(t) =', round(p_t, 3), '  P(space) =', round(p_space, 3))

# 2.
p_vowel = 0.0
for v in 'aeiou':
    p_vowel += unigrams[v] / total_unigrams         # add the probability of this vowel
print('P(vowel) =', round(p_vowel, 3))

# 3.
p_not_space = 1 - p_space
print('P(not space) =', round(p_not_space, 3))

⭐ **Challenge (optional):** print the 10 most frequent characters. We build a list of `(count, character)` pairs and sort it from largest to smallest. Fill in the two blanks.

In [ ]:
pairs = []
for char, count in unigrams.items():
    pairs.append((count, char))
pairs = sorted(pairs, reverse=True)       # tuples sort by their first element: the count

for count, char in pairs[???]:            # the first 10 pairs
    print(f"'{char}'", count, round(???, 3))   # the probability of this character

In [ ]:
# Count each pair of neighbouring characters (a bigram).
# We need the position i, so we use an index-based loop.
# There is one bigram fewer than there are characters.
bigrams = {}
total_bigrams = len(chapter_one) - 1
for i in range(total_bigrams):
    pair = chapter_one[i:i+2]        # the character at position i and the one after it
    bigrams[pair] = bigrams.get(pair, 0) + 1

In [ ]:
bigrams

**Activity 4 (conditional probability)**

The bigram counts tell us how likely a character is **given** the one before it:

$$P(h \mid t) = \frac{\text{count of } th}{\text{count of } t}$$

The first two lines work this out as an example.

1. Work out P(h | s) in the same way.
2. Work out P(h), the plain probability of `h` (as in Activity 3).
3. Compare the three numbers. Is `h` much more likely after `t`? What does this tell us about whether letters are independent of each other?

In [ ]:
p_h_given_t = bigrams['th'] / unigrams['t']
print('P(h | t) =', round(p_h_given_t, 3))

# 1.
p_h_given_s = bigrams["sh"] / unigrams['s']
print('P(h | s) =', round(p_h_given_s, 3))

# 2.
p_h = unigrams['h'] / total_unigrams
print('P(h)     =', round(p_h, 3))

Which characters can follow `q`? This loop looks at every bigram and prints those whose first character is `q`, with P(next | q). Fill in the blank and run it. What do you notice?

In [ ]:
for pair, count in bigrams.items():
    if pair[0] == 'q':
        print(f"'{pair[1]}'", round(count / unigrams['q'], 3))

We can now use the counts to calculate the probability of a sentence, including sentences that weren't in the text the model was built from. We add up the negative log probabilities (the *surprisal*) of each character given the one before it:

In [ ]:
# math is a library of mathematical functions that comes with Python. 'import math' loads it.
import math
# math.log(x, 2) is the logarithm of x to base 2
# The sentence is a string, so each element is a character.
# It starts with a space, which marks the start of the first word.
sentence = " the cat on the mat and ate his dinner"
log_prob = 0.0
prob = 1.0
for i in range(len(sentence) - 1):
    pair = sentence[i:i+2]
    p = bigrams.get(pair, 0) / unigrams[pair[0]]      # P(next character | this character)
    log_prob += -math.log(p, 2)
    prob *= p
log_prob
#prob

## The Shannon game

Shannon (1951) described an ingenious way of estimating the entropy of English by using human predictions. A person guesses a text one character at a time, and the number of guesses they need for each character tells us how predictable English is.

Shannon, C. E. (1951). Prediction and entropy of printed English. Bell system technical journal, 30(1), 50-64.

**Play it:** run the cell below (click ▶). It shows the text so far and asks you to guess the next character: a letter, or the word `space`. Keep guessing until you get it right, and the game moves on to the next character. At the end it shows how many guesses each character took and turns this into an estimate of the entropy of English, in bits per character.

Compare your estimate with your neighbours'. Keep a note of it: next we will estimate the entropy of English from our character model, and compare the two.

In [ ]:
#@title ▶ Run this cell to play the Shannon game { display-mode: "form" }
# (Double-click here if you want to see how the game works.)
import math

sg_text = "the old dog slept by the fire all afternoon"
sg_alphabet = "abcdefghijklmnopqrstuvwxyz "

print("Guess the text one character at a time.")
print("Type a letter (a-z), or type 'space' for a space, then press Enter.")
print("Type 'quit' to stop early.\n")

sg_guesses = []          # how many guesses each character took
sg_quit = False
for sg_pos in range(len(sg_text)):
    sg_target = sg_text[sg_pos]
    sg_tried = []
    while True:
        sg_reply = input(sg_text[:sg_pos] + "_    guess " + str(len(sg_tried) + 1) + ": ").lower()
        if sg_reply == "quit":
            sg_quit = True
            break
        if sg_reply == "space":
            sg_reply = " "
        if len(sg_reply) != 1 or sg_reply not in sg_alphabet:
            print("   Please type a single letter, or the word 'space'.")
            continue
        if sg_reply in sg_tried:
            print("   You have already tried that one.")
            continue
        sg_tried.append(sg_reply)
        if sg_reply == sg_target:
            print("   ✓ correct!")
            break
        else:
            print("   ✗ no")
    if sg_quit:
        break
    sg_guesses.append(len(sg_tried))

print()
print("Text so far:", sg_text[:len(sg_guesses)])
if len(sg_guesses) > 0:
    # how often did the right answer come on the 1st guess, the 2nd guess, ...?
    sg_counts = {}
    for g in sg_guesses:
        sg_counts[g] = sg_counts.get(g, 0) + 1
    print("Number of guesses needed : how many characters")
    for g in sorted(list(sg_counts.keys())):
        print("  ", g, "guess(es) :", sg_counts[g])

    # Shannon's estimate: the entropy of the number-of-guesses distribution
    n = len(sg_guesses)
    shannon_upper = 0.0
    for g in sg_counts:
        q = sg_counts[g] / n
        shannon_upper += -q * math.log(q, 2)

    print()
    print("Average number of guesses per character:", round(sum(sg_guesses) / n, 2))
    print("Your estimate of the entropy of English: at most", round(shannon_upper, 2), "bits per character")
    print("(This is Shannon's upper bound. The better your guesses, the lower it gets.)")

**Optional: the full version at home.** The original Shannon game program lets you play with longer texts. To play it in your own time:

Copy this URL into a browser window and press return

https://github.com/lianghuang3/shannon_game/archive/refs/heads/main.zip

Unpack repository to somewhere on local machine

If you are on University Computer or have VS Code installed
Open VSCode (e.g. via Anaconda Navigator)
Select File -> Open Folder and then select the shannon_game folder

Open shannon_game.py and press Run

Follow instructions given by the program once running.

## Calculate Entropy of English from a sample text

How do the estimates from our character model compare with your Shannon game estimate?

Now we want to use these counts to calculate the entropy of English.

If we were to calculate the entropy assuming every character was independent and occurred an equal number of times, the calculation would be very simple:

In [ ]:
import math
# math.log(x, 2) is the logarithm of x to base 2
-math.log(1/27, 2)

We know that letters do not occur the same number of times (are not equiprobable) so the next thing we might try is to calculate the entropy reflecting the unequal rates but still assuming independence.

In [ ]:
import math
# Entropy: add up -p * log2(p) over every character. math.log(p, 2) is log to base 2.
H = 0.0
for ch, count in unigrams.items():
    p = count / total_unigrams       # how often this character occurs
    H += -p * math.log(p, 2)
H

We know that characters are not independent - for example, h is more likely to occur following t than following x. This is why we use bigram probabilities.

In [ ]:
H = 0.0
for pair, count in bigrams.items():
    p_pair = count / total_bigrams          # P(x, y): how often this pair occurs
    p_next = count / unigrams[pair[0]]      # P(y | x): how often y follows x
    H += -p_pair * math.log(p_next, 2)
H

How does this estimate compare with your Shannon game estimate? Shannon's own experiments suggested that English carries roughly 1 bit per character or less. Why might our bigram model's estimate be so much higher?